# 改用国内多模态模型通义千问 Qwen3-VL-Plus

In [1]:
#!/usr/bin/env python
# coding: utf-8
import os
import time
import base64
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

In [2]:

# -------------------------- 初始化通义千问VL客户端 --------------------------
# 加载密钥
load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")
if not api_key:
    raise Exception("请在.env文件配置DASHSCOPE_API_KEY阿里云密钥")

# 兼容OpenAI接口
client = OpenAI(
    api_key=api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
MODEL_NAME = "qwen3-vl-plus"

In [3]:
# -------------------------- 工具函数：图片转base64 --------------------------
def image_to_base64(img_path: str) -> str:
    path = Path(img_path)
    suffix = path.suffix.lower()
    mime_type = "image/jpeg"
    if suffix == ".png":
        mime_type = "image/png"
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{data}"

In [4]:
# -------------------------- 工具函数：本地视频转base64（通义千问支持短视频） --------------------------
def video_to_base64(video_path: str) -> str:
    path = Path(video_path)
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return f"data:video/mp4;base64,{data}"

In [5]:
# ==================== 1. 纯文字问答（对应原Gemini文本输出） ====================
print("===== 文本问答测试 =====")
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "用中文解释AI大模型是如何工作的"
        }
    ],
    temperature=0.1
)
print(response.choices[0].message.content)
print("\n" + "-"*60 + "\n")

===== 文本问答测试 =====
AI大模型（如GPT、LLaMA、Qwen等）本质上是一种**基于深度学习的神经网络模型**，其核心目标是：**通过大量数据学习语言的统计规律，从而生成连贯、合理、符合语境的文本或完成各类任务**。下面我用通俗易懂的方式分步骤解释它的工作原理：

---

### 一、核心思想：**“预测下一个词”**
大模型的本质是一个**超大规模的语言概率预测器**。  
它不真正“理解”语言，而是通过海量文本训练，学会：
> “在给定前面一串词（上下文）的情况下，**最可能接什么词**？”

例如输入：“今天天气很好，我想去……”，模型会根据训练数据中类似句子的统计规律，预测出“公园散步”、“爬山”、“晒太阳”等高概率选项，并从中采样输出。

---

### 二、关键技术组成

#### 1. **Transformer 架构**（核心骨架）
- 2017年提出的革命性结构，取代了早期的RNN/LSTM。
- 关键机制：**自注意力机制（Self-Attention）**
  - 允许模型在处理一个词时，**动态关注句子中所有其他词的重要性**（比如“它”指代哪个名词）。
  - 举例：  
    > “小明把球扔给了小红，她接住了。”  
    模型需知道“她”=小红，而非小明——自注意力能建模这种长距离依赖。

- Transformer由**编码器（Encoder）** 和/或 **解码器（Decoder）** 组成：
  - GPT系列是**纯解码器架构**（适合生成任务）；
  - BERT是**纯编码器架构**（适合理解任务）；
  - 大模型多采用解码器（如GPT-4、Qwen）。

#### 2. **预训练（Pre-training）**：从无到有学语言
- 在**海量无标注文本**（如网页、书籍、代码等）上训练。
- 常用任务：**自回归语言建模**（Autoregressive LM）  
  → 输入前N个词，预测第N+1个词；不断滑动窗口训练。
- 模型参数量巨大（如GPT-3有1750亿参数），相当于“记忆”了语言的复杂模式。

#### 3. **微调（Fine-tuning）与对齐（Alignment）**
- 预训练后模型“会说话但不听话”——可能胡说、有害、不遵循指令。
- 通过以下方式优化：
 

In [6]:

# ==================== 2. 图片理解（对应原dog_and_girl.jpeg图像解析） ====================
print("===== 图片理解测试 =====")
img_base64 = image_to_base64("dog_and_girl.jpeg")
content_list = [
    {"type": "text", "text": "帮我解释下这张照片"},
    {"type": "image_url", "image_url": {"url": img_base64}}
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": content_list}],
    temperature=0.1
)
print(response.choices[0].message.content)
print("\n" + "-"*60 + "\n")

===== 图片理解测试 =====
这张照片捕捉了一个温暖、宁静而充满情感的瞬间——一位年轻女性和一只金毛犬在海滩上互动，背景是柔和的日落（或日出）光线。

以下是画面的详细解读：

🔹 **主体人物与动物**  
- 一位长发女性坐在沙滩上，身穿格子衬衫和深色裤子，赤脚，面带灿烂笑容，眼神温柔地注视着狗狗。她的姿态放松自然，流露出喜悦与亲密感
- 一只浅金色的拉布拉多犬（或金毛寻回犬）端坐于她对面，前爪抬起，正与她“击掌”（paw shake），动作乖巧又充满信任。它佩戴着彩色图案的胸背带和牵引绳，说明是家养宠物，且主人注重安全

🔹 **场景与氛围**  
- 地点：海边沙滩，沙粒细腻，可见轻微脚印与波浪冲刷痕迹  
- 时间：黄金时刻（Golden Hour）——太阳接近地平线，光线温暖柔和，呈橙金色调，从右侧（画面外）洒下，在人物轮廓和狗毛上形成逆光光晕，营造出梦幻、治愈的氛围  
- 背景：平静的海面泛着微光，远处有轻柔的浪花，天空明亮但无强烈对比，整体构图简洁开阔

🔹 **情感与象征意义**  
- 这是一幅典型的“人与宠物情感联结”影像：击掌动作象征合作、信任与默契；女子的笑容传递纯粹的快乐；狗狗专注的眼神体现忠诚与依恋  
- 画面隐含的主题包括：陪伴、治愈、自然中的宁静时光、简单生活的美好  
- 没有其他干扰元素（如人群、建筑），强化了“二人（一狗）世界”的私密与珍贵感

🔹 **摄影技巧亮点**  
- 低角度拍摄，增强亲和力与沉浸感  
- 浅景深（背景虚化），突出主体互动  
- 逆光+侧光结合，勾勒轮廓并渲染情绪  
- 色彩和谐：暖调（阳光、皮肤、狗毛）与冷调（沙、海）平衡，视觉舒适

✅ 总结：  
这不仅是一张“人与狗在海边玩耍”的照片，更是一则关于**爱、陪伴与当下幸福**的视觉诗篇。它唤起观者对简单快乐的共鸣——无需喧嚣，只需一个微笑、一次击掌，和一片海风拂面的黄昏。

如果你是在某处看到这张图（比如广告、社交媒体或摄影集），它很可能被用于传达品牌温度（如宠物用品、旅行、心理健康等），因为它极具感染力与普世价值 🌅🐾✨

------------------------------------------------------------



In [7]:
# ==================== 3. 短视频理解（对应原car.mp4视频解析） ====================
# 注意限制：通义千问VL本地视频base64仅支持短时长MP4（≤1分钟，≤20MB）
print("===== 视频理解测试 =====")
print("读取本地视频文件...")
video_base64 = video_to_base64("car.mp4")

content_list = [
    {"type": "text", "text": "详细描述视频里发生了什么？如果有对话，请把关键对话提取出来。"},
    {"type": "video_url", "video_url": {"url": video_base64}}
]

print("开始推理视频内容...")
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": content_list}],
    temperature=0.1
)
print(response.choices[0].message.content)

===== 视频理解测试 =====
读取本地视频文件...
开始推理视频内容...
视频内容详细描述如下：

**整体事件**：一辆白色SUV在地下车库转弯时，车门与墙壁发生碰撞，导致车身严重刮擦变形。车主（一名戴眼镜的男子）与妻子现场检查损伤，并讨论维修方案——因未投保车损险，只能自费修复。

---

### **分段详解**

#### **0:00–0:06**  
- 镜头特写白色SUV右后轮拱区域：  
  - 黑色轮眉饰条大面积剥落、撕裂，露出底下的灰色基材；  
  - 白色车漆被刮出多道长条状划痕，部分区域已露底漆或腻子层（呈灰白色）；  
  - 轮胎侧壁也有轻微蹭痕。  
- 字幕出现：**“直接撞墙上了”**  
  → 表明事故为车辆直接撞击墙体所致。

#### **0:07–0:14**  
- 镜头拉远，显示男子（穿深灰印有“CAT-DESIGNS”图案T恤+黑紫运动短裤）蹲在车旁检查损伤。  
- 他用手触摸并比划受损部位，表情略带无奈但语气平静。  
- 字幕依次出现：  
  - **“车门都给撞变形了”**  
  - **“下地下车库的时候”**  
  - **“门撞墙上了”**  
  - **“整个这个门都凹回去了”**  
  - **“下面的漆都掉了”**  
  → 说明事故发生在地下车库，撞击点位于车门与轮拱连接处，导致车门局部内凹、漆面脱落。

#### **0:15–0:21**  
- 近景特写男子手指沿刮痕滑动，从轮眉上沿延伸至车门下边缘（靠近门把手下方）。  
- 字幕：  
  - **“从这块到这块”**  
  - **“到门把手这块”**  
  - **“这块整个门都凹回去了”**  
  → 强调损伤范围广，涉及轮眉、侧裙及车门主体结构变形。

#### **0:22–0:26**  
- 女子（戴金丝边眼镜、白T恤、发髻插花饰）加入画面，蹲在男子身旁共同查看。  
- 她转向镜头提问，男子回应：  
  - 女子：“**老公你说咱们这走保险呀？还是自己修呀？**”  
  - 男子：“**咱们没有车损险**”  
  - 女子追问（字幕未显，但口型可辨）→ 男子补充：  
    - **“咱只能自己修了”**  
  → 关键信息：该车未投保车损险，无法通过保险理赔，需自费维修

#